# 04 · Drift Analysis
Compare fare_amount distribution: 2022 (reference) vs 2023 (current).

In [ ]:
import numpy as np
from pyspark.sql import functions as F
from src.common.config import get_settings
from src.aiops.drift_detector import detect_drift
from src.aiops.monitor import record_drift_score

settings = get_settings()
ws, lh = settings.fabric_workspace_id, settings.lakehouse_name
silver_path = f'abfss://{ws}@onelake.dfs.fabric.microsoft.com/{lh}.Lakehouse/Tables/silver_nyctaxi'

df = spark.read.format('delta').load(silver_path)

ref = np.array(df.filter(F.year('pickup_ts')==2022).select('fare_amount').dropna()
               .rdd.map(lambda r: float(r[0])).collect(), dtype=np.float64)
cur = np.array(df.filter(F.year('pickup_ts')==2023).select('fare_amount').dropna()
               .rdd.map(lambda r: float(r[0])).collect(), dtype=np.float64)

report = detect_drift('fare_amount', ref, cur,
                       psi_threshold=settings.drift_psi_threshold,
                       ks_threshold=settings.drift_ks_threshold)
print(report)
record_drift_score('fare_amount', report.psi, report.ks_statistic, report.wasserstein)